In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import warnings, time, json
from pathlib import Path
from gurobipy import Model, GRB, quicksum
import scipy.stats as st
warnings.filterwarnings('ignore')


In [ ]:

BASE = Path(os.environ.get("KEP_DATA_DIR", "../../data"))
POOL_DIR     = BASE / 'supplementary' / 'pool_simulations_databalance_full'
MATRICES_DIR = BASE / 'supplementary' / 'pool_matrices_databalance_full'
RESULTS_DIR  = BASE / 'supplementary' / 'simulation_results'
POOL_DIR.mkdir(parents=True, exist_ok=True)
MATRICES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SIMS = 100
LOCI = ['A', 'B', 'C', 'DR', 'DQ']
EPLET_CLASSES = ['ClassI', 'DR', 'DQ']
ETHCATS = [1, 2, 4, 5, 6, 7]
ETH_LABELS = {1: 'Caucasian', 2: 'Afroamerican', 4: 'Latin', 5: 'Asian', 6: 'AmInd', 7: 'PacIsl'}


PATIENT_TARGETS = {1: 300, 2: 300, 4: 300, 5: 100}  # = 1000 total

DONOR_TARGETS = {1: 600, 2: 66, 4: 300, 5: 32, 6: 5}  


ARRIVAL_RATE = 1000 / (10 * 12)  

#Load and clean source data
df_pat_all = pd.read_csv(BASE / 'df_receptores_imputados_final.csv', low_memory=False)
df_don_all = pd.read_csv(BASE / 'df_donantes_imputados_final.csv', low_memory=False)
df_pat_all['WL_ID_CODE'] = df_pat_all['WL_ID_CODE'].astype('int64')
df_pat_all['ETHCAT'] = pd.to_numeric(df_pat_all['ETHCAT'], errors='coerce')
df_don_all['ETHCAT_DON'] = pd.to_numeric(df_don_all['ETHCAT_DON'], errors='coerce')

print('PATIENT TARGETS (balanced):')
for e in (1, 2, 4, 5):
    npat = (df_pat_all['ETHCAT'] == e).sum()
    print(f'  {ETH_LABELS[e]:13s}: {npat:4d} available, target {PATIENT_TARGETS[e]}')

print('\nDONOR TARGETS (balanced — oversample minorities):')
for e in (1, 2, 4, 5, 6):
    ndon = (df_don_all['ETHCAT_DON'] == e).sum()
    tgt = DONOR_TARGETS.get(e, 0)
    print(f'  {ETH_LABELS[e]:13s}: {ndon:4d} available, target {tgt}')
print(f'  TOTAL donors: {len(df_don_all)}')

SIM_PARAMS = {
    'TOTAL_TIME':       10 * 12,
    'ARRIVAL_RATE':     ARRIVAL_RATE,
    'MEAN_PATIENCE':    65.1552,   
    'MATCH_RUN':        3,
    'WARMUP_MONTHS':    0,   
    'MAX_CYCLE_LENGTH': 3,
    'SEED_BASE':        42,
    'P':                1000,
    'k_opt': {'antigen': 0, 'allele': 0, 'eplet': 0},
    'Z':     {'antigen': 10, 'allele': 10, 'eplet': 140},
}
print(f"\nSIM_PARAMS:")
print(f'  ARRIVAL_RATE (global, /month): {SIM_PARAMS["ARRIVAL_RATE"]:.4f}')
print(f'  Total expected arrivals:    {SIM_PARAMS["ARRIVAL_RATE"] * SIM_PARAMS["TOTAL_TIME"]:.0f}')


WARMUP_SUFFIX = '_warmup' if SIM_PARAMS.get('WARMUP_MONTHS', 0) > 0 else '_nowarmup'
print(f"\n>>> VERSION: WARMUP_MONTHS={SIM_PARAMS['WARMUP_MONTHS']} -> outputs with suffix '{WARMUP_SUFFIX}'")

RESOLUTIONS = ('antigen', 'allele', 'eplet')


In [ ]:

import re

def parse_antibody(code):
    s = str(int(code)); n = len(s)
    if n <= 2: return ('antigen', s.zfill(2))
    elif n == 3: return ('allele', f'0{s[0]}:{s[1:]}')
    elif n == 4: return ('allele', f'{s[:2]}:{s[2:]}')
    elif n == 5: return ('allele', f'0{s[0]}:{s[1:3]}')
    else: return ('allele', f'{s[:2]}:{s[2:4]}')

df_unacc = pd.read_parquet(BASE / 'unacc_filtered.parquet')
antibodies = {}
for wid, group in df_unacc.groupby('WL_ID_CODE'):
    s = set()
    for ant, loc in zip(group['ANTIGEN'].astype(int).values, group['LOCUS'].values):
        level, value = parse_antibody(ant)
        s.add((loc, level, value))
    antibodies[int(wid)] = s
print(f'Antibodies loaded for {len(antibodies)} patients')

_ABO_COMPAT = {('O','O'),('O','A'),('O','B'),('O','AB'),('A','A'),('A','AB'),('B','B'),('B','AB'),('AB','AB')}
def abo_compatible(d_abo, p_abo): return (d_abo, p_abo) in _ABO_COMPAT

PAT_COLS = {'A': ('A1_pat','A2_pat'), 'B': ('B1_pat','B2_pat'), 'C': ('C1_pat','C2_pat'),
            'DR': ('DR1_pat','DR2_pat'), 'DQ': ('DQ1_pat','DQ2_pat')}
DON_COLS = {'A': ('DA1','DA2'), 'B': ('DB1','DB2'), 'C': ('DC1','DC2'),
            'DR': ('DDR1','DDR2'), 'DQ': ('DDQ1','DDQ2')}

N_IMPUTATIONS = 45
def parse_imputation_string(imp_str):
    result = {f'{L}{n}': None for L in ('A','B','C','DR','DQ') for n in (1,2)}
    if not isinstance(imp_str, str): return result
    tokens = imp_str.replace('+', '').split()
    locus_alleles = {'A': [], 'B': [], 'C': [], 'DR': [], 'DQ': []}
    for tok in tokens:
        if '*' not in tok: continue
        locus_raw, allele = tok.split('*', 1)
        if locus_raw == 'A': locus_alleles['A'].append(allele)
        elif locus_raw == 'B': locus_alleles['B'].append(allele)
        elif locus_raw == 'C': locus_alleles['C'].append(allele)
        elif locus_raw in ('DRB1', 'DR'): locus_alleles['DR'].append(allele)
        elif locus_raw in ('DQB1', 'DQ'): locus_alleles['DQ'].append(allele)
    for L in ('A','B','C','DR','DQ'):
        if len(locus_alleles[L]) >= 1: result[f'{L}1'] = locus_alleles[L][0]
        if len(locus_alleles[L]) >= 2: result[f'{L}2'] = locus_alleles[L][1]
        elif len(locus_alleles[L]) == 1: result[f'{L}2'] = locus_alleles[L][0]
    return result

def sample_hla_imputation(row, rng):
    imps, liks = [], []
    for i in range(1, N_IMPUTATIONS + 1):
        imp = row.get(f'Imputacion_{i}')
        lik = row.get(f'Likelihood_{i}')
        if pd.notna(imp) and pd.notna(lik):
            imps.append(imp); liks.append(float(lik))
    if not imps: return None
    weights = np.array(liks) / sum(liks)
    idx = rng.choice(len(imps), p=weights)
    return parse_imputation_string(imps[idx])

def fixed_hla_patient(row):
    return {f'{L}{n}': row.get(f'{L}{n}') for L in ('A','B','C','DR','DQ') for n in (1,2)}

def fixed_hla_donor(row):
    return {f'{L}{n}': row.get(f'D{L}{n}' if L != 'A' else f'DA{n}') for L in ('A','B','C','DR','DQ') for n in (1,2)}

def assign_hla_to_pair_row(pair_dict, hla_dict, side='pat'):
    col_map = PAT_COLS if side == 'pat' else DON_COLS
    for L, (c1, c2) in col_map.items():
        pair_dict[c1] = hla_dict.get(f'{L}1')
        pair_dict[c2] = hla_dict.get(f'{L}2')

def donor_dsa_triggers_from_hla(hla_dict):
    out = set()
    for L in ('A','B','C','DR','DQ'):
        for n in (1, 2):
            v = hla_dict.get(f'{L}{n}')
            if pd.isna(v) or v in (None, '', 'nan'): continue
            full = str(v)
            ff = full.split(':')[0].zfill(2)
            out.add((L, 'antigen', ff))
            out.add((L, 'allele', full))
    return out

def donor_dsa_triggers(row_don):
    out = set()
    for L, (c1, c2) in DON_COLS.items():
        for c in (c1, c2):
            v = row_don.get(c)
            if pd.isna(v) or v in (None, '', 'nan'): continue
            full = str(v)
            ff = full.split(':')[0].zfill(2)
            out.add((L, 'antigen', ff))
            out.add((L, 'allele', full))
    return out

def hla_pat_str(row):
    PMAP = {'A':'A','B':'B','C':'C','DR':'DRB1','DQ':'DQB1'}
    parts = []
    for L, (c1, c2) in PAT_COLS.items():
        for c in (c1, c2):
            v = row.get(c)
            if pd.notna(v) and v not in ('','nan'): parts.append(f'{PMAP[L]}*{v}')
    return ' '.join(parts)

def hla_don_str(row):
    PMAP = {'A':'A','B':'B','C':'C','DR':'DRB1','DQ':'DQB1'}
    parts = []
    for L, (c1, c2) in DON_COLS.items():
        for c in (c1, c2):
            v = row.get(c)
            if pd.notna(v) and v not in ('','nan'): parts.append(f'{PMAP[L]}*{v}')
    return ' '.join(parts)


In [ ]:
# POOL FORMATION: balanced patients + BALANCED donors (oversample minorities) + per-slot HLA sampling

def sample_patients_balanced(rng):
   
    slots = []
    for eth, target in PATIENT_TARGETS.items():
        pool_eth = df_pat_all[df_pat_all['ETHCAT'] == eth].reset_index(drop=True)
        if len(pool_eth) == 0: continue
        if target <= len(pool_eth):
            idx = rng.choice(len(pool_eth), size=target, replace=False)
        else:
            
            idx = list(range(len(pool_eth))) + list(rng.integers(0, len(pool_eth), size=target - len(pool_eth)))
            rng.shuffle(idx)
        for i in idx:
            row = pool_eth.iloc[i]
            hla = sample_hla_imputation(row, rng)
            if hla is None: hla = fixed_hla_patient(row)
            slots.append({
                'WL_ID_CODE': int(row['WL_ID_CODE']),
                'ETHCAT': int(eth),
                'ABO_pat': row['ABO'],
                'antibodies': antibodies.get(int(row['WL_ID_CODE']), set()),
                'hla': hla,
            })
    rng.shuffle(slots)
    for k, s in enumerate(slots): s['slot_id'] = k
    return slots

def sample_donors_balanced(rng):
    """Oversample minority donors and sub-sample Caucasians per DONOR_TARGETS"""
    slots = []
    for eth, target in DONOR_TARGETS.items():
        pool_eth = df_don_all[df_don_all['ETHCAT_DON'] == eth].reset_index(drop=True)
        if len(pool_eth) == 0: continue
        if target <= len(pool_eth):
            idx = rng.choice(len(pool_eth), size=target, replace=False)
        else:
            # Oversample with replacement
            idx = list(range(len(pool_eth))) + list(rng.integers(0, len(pool_eth), size=target - len(pool_eth)))
            rng.shuffle(idx)
        for i in idx:
            row = pool_eth.iloc[i]
            hla = sample_hla_imputation(row, rng)
            if hla is None: hla = fixed_hla_donor(row)
            triggers = donor_dsa_triggers_from_hla(hla)
            slots.append({
                'DONOR_ID': int(row.get('DONOR_ID', 0) or 0),
                'ETHCAT_DON': int(row['ETHCAT_DON']) if pd.notna(row['ETHCAT_DON']) else -1,
                'ABO_don': row['ABO_DON'],
                'hla': hla,
                'triggers': triggers,
            })
    rng.shuffle(slots)
    for k, s in enumerate(slots): s['slot_id'] = k
    return slots

def is_incompatible(p_abo, d_abo, p_ab, d_trig):
    if pd.isna(p_abo) or pd.isna(d_abo): return False
    if not abo_compatible(d_abo, p_abo): return True
    if p_ab and (d_trig & p_ab): return True
    return False

def form_pool_balanced(sim_id):
    rng = np.random.default_rng(200 + sim_id)
    don_slots = sample_donors_balanced(rng)
    pat_by_eth = {e: df_pat_all[df_pat_all['ETHCAT'] == e].reset_index(drop=True)
                  for e in PATIENT_TARGETS}

    demand = [e for e, t in PATIENT_TARGETS.items() for _ in range(t)]
    rng.shuffle(demand)

    pairs = []
    used_dons = set()
    MAX_TRIES = 3000
    for e in demand:
        pool_eth = pat_by_eth[e]
        for _try in range(MAX_TRIES):
            row = pool_eth.iloc[int(rng.integers(0, len(pool_eth)))]
            p_abo = row['ABO']
            p_ab  = antibodies.get(int(row['WL_ID_CODE']), set())
            matched = False
            for ds in don_slots:
                if ds['slot_id'] in used_dons:
                    continue
                if is_incompatible(p_abo, ds['ABO_don'], p_ab, ds['triggers']):
                    p_hla = sample_hla_imputation(row, rng)
                    if p_hla is None:
                        p_hla = fixed_hla_patient(row)
                    pair = {
                        'WL_ID_CODE':  int(row['WL_ID_CODE']),
                        'DONOR_ID':    ds['DONOR_ID'],
                        'pat_slot_id': len(pairs),
                        'don_slot_id': ds['slot_id'],
                        'ETHCAT':      int(e),
                        'ETHCAT_DON':  ds['ETHCAT_DON'],
                        'ABO_pat':     p_abo,
                        'ABO_don':     ds['ABO_don'],
                    }
                    assign_hla_to_pair_row(pair, p_hla, side='pat')
                    assign_hla_to_pair_row(pair, ds['hla'], side='don')
                    pairs.append(pair)
                    used_dons.add(ds['slot_id'])
                    matched = True
                    break
            if matched:
                break
        else:
            raise RuntimeError(f"sim {sim_id}: no pair for eth {e} (donors {len(used_dons)}/{len(don_slots)})")
    return pd.DataFrame(pairs)


In [ ]:
# GENERATE 100 SIM POOLS

OVERWRITE_POOLS = False

t0 = time.time()
for sim_id in range(N_SIMS):
    out_path = POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet'
    if out_path.exists() and not OVERWRITE_POOLS:
        continue
    pool_df = form_pool_balanced(sim_id)
    pool_df.to_parquet(out_path, compression='snappy')
    if (sim_id + 1) % 10 == 0 or sim_id == 0:
        
        comp = pool_df['ETHCAT'].value_counts().sort_index().to_dict()
        print(f'  sim {sim_id+1:3d}/{N_SIMS}: pool size {len(pool_df)} — by ETHCAT: {comp}  ({(time.time()-t0)/60:.1f} min elapsed)')

print(f'\nAll pools generated. Total time: {(time.time()-t0)/60:.1f} min')


print('\n━━━ Pool composition summary across 100 sims ━━━')
all_sizes = []; all_comp = []
for sim_id in range(N_SIMS):
    df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    all_sizes.append(len(df))
    comp = df['ETHCAT'].value_counts().sort_index().to_dict()
    all_comp.append(comp)
print(f'Pool size — mean: {np.mean(all_sizes):.0f}, std: {np.std(all_sizes):.1f}')
for e in (1, 2, 4, 5):
    counts = [c.get(e, 0) for c in all_comp]
    print(f'  {ETH_LABELS[e]:13s}: mean {np.mean(counts):.0f}, min {min(counts)}, max {max(counts)}  (target {PATIENT_TARGETS[e]})')


In [ ]:
# PRECOMPUTE MATRICES (compat + antigen/allele/eplet per locus)

OVERWRITE_MATRICES = False

import os, sys
HANS_DIR = BASE / 'codigo_Hans_2' / 'ET_eplet_calculator-master'
src_yaml = HANS_DIR / 'simulator/sim_yamls/sim_settings_test_eplet.yaml'
patched_yaml = HANS_DIR / 'simulator/sim_yamls/_patched_settings.yaml'
with open(src_yaml) as f: text = f.read()
old_prefix = '/Users/vale/Library/CloudStorage/OneDrive-UniversidadAdolfoIbanez/TESIS/CODIGO/MISMATCH EPLET EPREGISTRY/codigo_Hans_2/ET_eplet_calculator-master/'
new_prefix = str(HANS_DIR) + '/'
patched_yaml.write_text(text.replace(old_prefix, new_prefix))

orig_cwd = os.getcwd()
os.chdir(HANS_DIR)
if str(HANS_DIR) not in sys.path: sys.path.insert(0, str(HANS_DIR))

import simulator.magic_values.etkidney_simulator_settings as es
from simulator.code.HLA.HLASystem import HLASystem, HLAProfile
from simulator.code.HLA.EpletSystem import EpletSystem
import simulator.code.utils.read_input_files as rdr
import simulator.magic_values.magic_values_rules as mgr

ss = rdr.read_sim_settings(str(patched_yaml))
ss.NEEDED_SPLIT_MISMATCHES = [mgr.HLA_A, mgr.HLA_B, mgr.HLA_C, mgr.HLA_DR, mgr.HLA_DQB]
ss.NEEDED_BROAD_MISMATCHES = [mgr.HLA_A, mgr.HLA_B, mgr.HLA_C, mgr.HLA_DR, mgr.HLA_DQB]
print('Initializing HLA + Eplet systems...')
hla_system = HLASystem(ss)
eplet_system = EpletSystem(ss, hla_system)
os.chdir(orig_cwd)
print('  done')

LOCUS_PREFIX = {'A': 'A', 'B': 'B', 'C': 'C', 'DR': 'DRB1', 'DQ': 'DQB1'}

def allele_mm_locus(d_alleles, p_alleles):
    p_set = {str(x) for x in p_alleles if pd.notna(x) and str(x) != 'nan'}
    d_set = {str(x) for x in d_alleles if pd.notna(x) and str(x) != 'nan'}
    return sum(1 for d in d_set if d not in p_set)

def build_compat_matrix(sim_df):
    n = len(sim_df)
    pat_abo = sim_df['ABO_pat'].values
    don_abo = sim_df['ABO_don'].values
    pat_ids = sim_df['WL_ID_CODE'].values
    pat_antibodies = [antibodies.get(int(pat_ids[i]), set()) for i in range(n)]
    don_triggers = [donor_dsa_triggers(sim_df.iloc[i]) for i in range(n)]
    compat = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(n):
            if i == j: continue
            if abo_compatible(don_abo[j], pat_abo[i]) and not (don_triggers[j] & pat_antibodies[i]):
                compat[i, j] = 1
    return compat

def build_allele_mm_per_locus(sim_df, compat):
    n = len(sim_df)
    out = {L: np.zeros((n, n), dtype=np.int8) for L in LOCI}
    for L in LOCI:
        c1p, c2p = PAT_COLS[L]; c1d, c2d = DON_COLS[L]
        pa = [[sim_df.iloc[i][c1p], sim_df.iloc[i][c2p]] for i in range(n)]
        da = [[sim_df.iloc[i][c1d], sim_df.iloc[i][c2d]] for i in range(n)]
        for i in range(n):
            for j in range(n):
                if compat[i, j] == 1:
                    out[L][i, j] = allele_mm_locus(da[j], pa[i])
    return out

def run_hans_on_compat(sim_df, compat):
    n = len(sim_df)
    pat_strings = [hla_pat_str(sim_df.iloc[i]) for i in range(n)]
    don_strings = [hla_don_str(sim_df.iloc[i]) for i in range(n)]
    pat_profiles = [HLAProfile(rdr.fix_hla_string(pd.Series([s]))[0], hla_system=hla_system) for s in pat_strings]
    don_profiles = [HLAProfile(rdr.fix_hla_string(pd.Series([s]))[0], hla_system=hla_system) for s in don_strings]
    antigen_keys = {'A':'mms_hla_a','B':'mms_hla_b','C':'mms_hla_c','DR':'mms_hla_dr','DQ':'mms_hla_dqb'}
    EPLET_CLASS_MAP = {'ClassI':'I','DR':'drb1345','DQ':'dq'}
    antigen_mat = {L: np.zeros((n, n), dtype=np.int8) for L in LOCI}
    eplet_mat   = {cls: np.zeros((n, n), dtype=np.int16) for cls in EPLET_CLASS_MAP}
    for i in range(n):
        ph = pat_profiles[i]
        js = np.where(compat[i] == 1)[0]
        for j in js:
            dh = don_profiles[j]
            try:
                mm = hla_system.count_mismatches(p_hla=ph, d_hla=dh)
                for L, key in antigen_keys.items():
                    v = mm.get(key, 0)
                    if pd.notna(v): antigen_mat[L][i, j] = int(v)
                ep_cls = eplet_system.get_epletregistry_mm_per_locus(pat_hla=ph, don_hla=dh)
                for cls_name, hans_key in EPLET_CLASS_MAP.items():
                    v = ep_cls.get(hans_key, 0)
                    if pd.notna(v): eplet_mat[cls_name][i, j] = int(v)
            except Exception:
                pass
    return antigen_mat, eplet_mat

def precompute_one_sim(sim_id, overwrite=False):
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    if sim_dir.exists() and not overwrite and (sim_dir / 'compatibility.parquet').exists():
        return False
    sim_dir.mkdir(parents=True, exist_ok=True)
    sim_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    compat = build_compat_matrix(sim_df)
    pd.DataFrame(compat).to_parquet(sim_dir / 'compatibility.parquet', compression='snappy')
    allele = build_allele_mm_per_locus(sim_df, compat)
    for L, mat in allele.items():
        pd.DataFrame(mat).to_parquet(sim_dir / f'mismatch_allele_{L}.parquet', compression='snappy')
    antigen, eplet = run_hans_on_compat(sim_df, compat)
    for L, mat in antigen.items():
        pd.DataFrame(mat).to_parquet(sim_dir / f'mismatch_antigen_{L}.parquet', compression='snappy')
    for cls, mat in eplet.items():
        pd.DataFrame(mat).to_parquet(sim_dir / f'mismatch_eplet_{cls}.parquet', compression='snappy')
    return True

t0 = time.time()
for sim_id in range(N_SIMS):
    did = precompute_one_sim(sim_id, overwrite=OVERWRITE_MATRICES)
    if did and ((sim_id + 1) % 5 == 0 or sim_id == 0):
        print(f'  sim {sim_id+1:3d}/{N_SIMS}: matrices done ({(time.time()-t0)/60:.1f} min)')
print(f'\nAll matrices done. Total time: {(time.time()-t0)/60:.1f} min')


In [ ]:
# PER-SIM DATA LOADER

def load_sim_data(sim_id):
    pool_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    compat = pd.read_parquet(sim_dir / 'compatibility.parquet').values.astype(np.int8)
    antigen_mm = {L: pd.read_parquet(sim_dir / f'mismatch_antigen_{L}.parquet').values for L in LOCI}
    allele_mm  = {L: pd.read_parquet(sim_dir / f'mismatch_allele_{L}.parquet').values for L in LOCI}
    eplet_mm   = {cls: pd.read_parquet(sim_dir / f'mismatch_eplet_{cls}.parquet').values for cls in EPLET_CLASSES}
    return {'pool_df': pool_df, 'compat': compat,
            'antigen_mm': antigen_mm, 'allele_mm': allele_mm, 'eplet_mm': eplet_mm}

sd = load_sim_data(0)
print(f"Sim 0 pool: {len(sd['pool_df'])} pairs, compat cells: {int(sd['compat'].sum()):,}")
print(f"Ethnicities: {sd['pool_df']['ETHCAT'].value_counts().sort_index().to_dict()}")


In [ ]:
# BUILD WEIGHTS (10 loci) + GRAPH + OPTIMIZATION 

MAX_ANTIGEN_10LOCI = 10; MAX_ALLELE_10LOCI = 10; MAX_EPLET_10LOCI = 140

def build_weights_10loci(sim_data):
    am = sim_data['antigen_mm']; al = sim_data['allele_mm']; ep = sim_data['eplet_mm']
    sum_antigen = sum(am[L] for L in LOCI).astype(np.int32)
    sum_allele  = sum(al[L] for L in LOCI).astype(np.int32)
    sum_eplet   = (ep['ClassI'] + ep['DR'] + ep['DQ']).astype(np.int32)
    return {
        'antigen': (MAX_ANTIGEN_10LOCI - sum_antigen).astype(np.int32),
        'allele':  (MAX_ALLELE_10LOCI  - sum_allele).astype(np.int32),
        'eplet':   (MAX_EPLET_10LOCI   - sum_eplet).astype(np.int32),
        'score_classI': (6 - (am['A'] + am['B'] + am['C'])).astype(np.int32),
        'score_DR':     (2 - am['DR']).astype(np.int32),
        'score_DQ':     (2 - am['DQ']).astype(np.int32),
    }

def create_graph(waiting_indices, compat):
    G = nx.DiGraph()
    G.add_nodes_from(waiting_indices)
    for i in waiting_indices:
        for j in waiting_indices:
            if i == j: continue
            if compat[i, j] == 1: G.add_edge(j, i)
    return G

def changing_resolution_weights(G, weight_matrix):
    for u, v in G.edges(): G[u][v]['weight'] = int(weight_matrix[v, u])

def optimization(G, l=3, k_quality=0, Z=10, P=1100):
    total_cycles = list(nx.simple_cycles(G, length_bound=l))
    valid_cycles = [c for c in total_cycles
                    if all(G[u][v]['weight'] >= k_quality
                           for u, v in zip(c, c[1:] + c[:1]))]
    G_opt = nx.DiGraph()
    if not valid_cycles: return G_opt, []
    m = Model('kep'); m.setParam('OutputFlag', 0)
    x = {tuple(c): m.addVar(vtype=GRB.BINARY) for c in valid_cycles}
    m.setObjective(
        quicksum(
            x[tuple(c)] * (
                (len(c) + (1.0 / P) * sum(G[u][v]['weight'] / Z
                                          for u, v in zip(c, c[1:] + c[:1])))
                / P
            )
            for c in valid_cycles
        ), GRB.MAXIMIZE)
    for node in G.nodes():
        m.addConstr(quicksum(x[tuple(c)] for c in valid_cycles if node in c) <= 1)
    m.optimize()
    selected = []
    if m.status == GRB.OPTIMAL:
        for c in valid_cycles:
            if x[tuple(c)].X > 0.5:
                selected.append(c)
                for i in range(len(c)):
                    u, v = c[i], c[(i + 1) % len(c)]
                    G_opt.add_edge(u, v, weight=G[u][v]['weight'])
    return G_opt, selected


In [ ]:
# RUN ONE SIMULATION 

def run_simulation(sim_id, opt_resolution, sim_data, weights, params):
    pool_df = sim_data['pool_df']
    compat  = sim_data['compat']
    n = len(pool_df)
    pair_ethcat = pool_df['ETHCAT'].values

    weight_for_obj = weights[opt_resolution]
    k_opt = params['k_opt'][opt_resolution]
    Z     = params['Z'][opt_resolution]

    
    ss = np.random.SeedSequence(params['SEED_BASE'] + sim_id * 1000)
    rng_arr, rng_dep = (np.random.default_rng(s) for s in ss.spawn(2))

    
    available = set(range(n))
    waiting = []
    arrival_t, departure_t = {}, {}
    historial_cycles = []
    historial_departures = []
    pool_sizes = []
    deadline = {}
    runs_participated = {}
    pool_sizes_by_eth = {e: [] for e in ETHCATS}

    arrivals_by_eth = {e: 0 for e in ETHCATS}
    departures_by_eth = {e: 0 for e in ETHCATS}
    quality = {(res, e): [] for res in ('antigen','allele','eplet') for e in ETHCATS}
    quality.update({(cls, e): [] for cls in ('classI','DR','DQ') for e in ETHCATS})

    # Each pair gets a uniform arrival month
    _arrival_month = rng_arr.integers(0, params['TOTAL_TIME'], size=n)
    _arrivals_at = {m: [] for m in range(params['TOTAL_TIME'])}
    for _p in range(n):
        _arrivals_at[int(_arrival_month[_p])].append(_p)

    WARMUP = params.get('WARMUP_MONTHS', 0)
    for month in range(params['TOTAL_TIME']):
        counting = month >= WARMUP
        for p_int in _arrivals_at[month]:
            arrival_t[p_int] = month
            available.discard(p_int)
            waiting.append(p_int)
            deadline[p_int] = month + rng_dep.exponential(params['MEAN_PATIENCE'])
            e = int(pair_ethcat[p_int])
            if counting and e in arrivals_by_eth: arrivals_by_eth[e] += 1

       
        if (month + 1) % params['MATCH_RUN'] == 0 and len(waiting) >= 2:
            if counting: pool_sizes.append(len(waiting))
            for e_ps in (ETHCATS if counting else []):
                pool_sizes_by_eth[e_ps].append(sum(1 for w in waiting if int(pair_ethcat[w]) == e_ps))
            for _w in waiting:
                runs_participated[_w] = runs_participated.get(_w, 0) + 1
            G = create_graph(waiting, compat)
            changing_resolution_weights(G, weight_for_obj)
            G_opt, selected = optimization(G, l=params['MAX_CYCLE_LENGTH'],
                                              k_quality=k_opt, Z=Z, P=params['P'])
            for u, v in (G_opt.edges() if counting else []):
                e = int(pair_ethcat[v])
                if e not in arrivals_by_eth: continue
                quality[('antigen', e)].append(int(weights['antigen'][v, u]))
                quality[('allele',  e)].append(int(weights['allele'][v, u]))
                quality[('eplet',   e)].append(int(weights['eplet'][v, u]))
                quality[('classI',  e)].append(int(weights['score_classI'][v, u]))
                quality[('DR',      e)].append(int(weights['score_DR'][v, u]))
                quality[('DQ',      e)].append(int(weights['score_DQ'][v, u]))
            historial_cycles.extend(selected if counting else [])
            cycled = {p for c in selected for p in c}
            waiting = [w for w in waiting if w not in cycled]
            for p_int in cycled: departure_t[int(p_int)] = month

        departed_now = [w for w in waiting if deadline[w] <= month]
        if departed_now:
            ds = set(departed_now)
            waiting = [w for w in waiting if w not in ds]
            for p_int in (departed_now if counting else []):
                historial_departures.append(p_int)
                e = int(pair_ethcat[p_int])
                if e in departures_by_eth:
                    departures_by_eth[e] += 1

    waiting_times_by_eth = {e: [] for e in ETHCATS}
    for p in {p for c in historial_cycles for p in c}:
        if p in runs_participated:
            e = int(pair_ethcat[p])
            if e in waiting_times_by_eth:
                waiting_times_by_eth[e].append(runs_participated[p])

    n_total_arr = sum(arrivals_by_eth.values())
    n_total_tx = sum(len(c) for c in historial_cycles)
    F_total = n_total_tx / max(n_total_arr, 1)
    L_total = len(historial_departures) / max(n_total_arr, 1)
    F_per_eth = {e: sum(1 for c in historial_cycles for p in c if int(pair_ethcat[p]) == e)
                       / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    L_per_eth = {e: departures_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}

    return {
        'sim_id': sim_id, 'opt_resolution': opt_resolution,
        'total_arrivals': n_total_arr, 'total_transplants': n_total_tx,
        'total_departures': len(historial_departures),
        'arrivals_by_eth': arrivals_by_eth, 'departures_by_eth': departures_by_eth,
        'F_per_eth': F_per_eth, 'L_per_eth': L_per_eth,
        'F_total': F_total, 'L_total': L_total,
        'quality': quality, 'waiting_times_by_eth': waiting_times_by_eth,
        'historial_cycles': historial_cycles,
        'avg_pool_size': float(np.mean(pool_sizes)) if pool_sizes else 0.0,
        'avg_pool_size_by_eth': {e: float(np.mean(pool_sizes_by_eth[e])) if pool_sizes_by_eth[e] else 0.0 for e in ETHCATS},
    }


In [ ]:
# FULL LOOP

import pickle
PICKLE_PATH = RESULTS_DIR / f"all_results_databalance_full_10loci{WARMUP_SUFFIX}.pkl"

if PICKLE_PATH.exists():
    with open(PICKLE_PATH, "rb") as _f:
        all_results = pickle.load(_f)
    print(f"Loaded cached all_results from {PICKLE_PATH.name} "
          f"({sum(len(v) for v in all_results.values())} sims total) — simulation skipped.")
else:
  

    all_results = {res: [] for res in RESOLUTIONS}
    t0 = time.time()
    for sim_id in range(N_SIMS):
        sim_data = load_sim_data(sim_id)
        weights = build_weights_10loci(sim_data)
        for opt_res in RESOLUTIONS:
            result = run_simulation(sim_id, opt_res, sim_data, weights, SIM_PARAMS)
            all_results[opt_res].append(result)
        if (sim_id + 1) % 5 == 0 or sim_id == 0:
            
            print(f'  Sim {sim_id+1:3d}/{N_SIMS}')
    


  
    with open(PICKLE_PATH, "wb") as _f:
        pickle.dump(all_results, _f)
    print(f"\nSaved all_results to {PICKLE_PATH.name} ({PICKLE_PATH.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# AGGREGATION + SAVE XLSX 

ACTIVE_ETHCATS = [1, 2, 4, 5]  

def mean_ci(values, conf=0.95):
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        if len(arr) == 1: return arr[0], f"{arr[0]:.3f} [-; -]"
        return float('nan'), 'nan'
    m, s = arr.mean(), arr.std(ddof=1)
    low, high = st.t.interval(conf, len(arr)-1, loc=m, scale=s/np.sqrt(len(arr)))
    return m, f"{m:.3f} [{low:.3f}; {high:.3f}]"

def build_results_table(results_for_res):
    rs = results_for_res
    rows = []
    for e in ACTIVE_ETHCATS:
        arr_e = np.mean([r['arrivals_by_eth'][e] for r in rs])
        tx_e = np.mean([r['F_per_eth'][e] * r['arrivals_by_eth'][e] for r in rs])

        F_vals = [r['F_per_eth'][e] for r in rs]
        L_vals = [r['L_per_eth'][e] for r in rs]
        _, txt_F = mean_ci(F_vals); _, txt_L = mean_ci(L_vals)

        ant_vals = [np.mean(r['quality'][('antigen', e)]) for r in rs if r['quality'][('antigen', e)]]
        all_vals = [np.mean(r['quality'][('allele',  e)]) for r in rs if r['quality'][('allele',  e)]]
        epl_vals = [np.mean(r['quality'][('eplet',   e)]) for r in rs if r['quality'][('eplet',   e)]]
        _, txt_ant = mean_ci(ant_vals); _, txt_all = mean_ci(all_vals); _, txt_epl = mean_ci(epl_vals)

        cI_vals = [np.mean(r['quality'][('classI', e)]) for r in rs if r['quality'][('classI', e)]]
        dr_vals = [np.mean(r['quality'][('DR', e)]) for r in rs if r['quality'][('DR', e)]]
        dq_vals = [np.mean(r['quality'][('DQ', e)]) for r in rs if r['quality'][('DQ', e)]]
        _, txt_cI = mean_ci(cI_vals); _, txt_dr = mean_ci(dr_vals); _, txt_dq = mean_ci(dq_vals)

        wt_flat = [w for r in rs for w in r['waiting_times_by_eth'][e]]
        wt_mean = np.mean(wt_flat) if wt_flat else float('nan')
        still = round(1 - np.mean(F_vals) - np.mean(L_vals), 3)

        rows.append({
            'Ethnicity': ETH_LABELS.get(e, str(e)),
            'Arrivals': round(arr_e, 2),
            'Transplants': round(tx_e, 2),
            'F(s) (Matched)': txt_F,
            'HLA(s) Antigen': txt_ant,
            'HLA(s) Allele': txt_all,
            'HLA(s) Eplets': txt_epl,
            'Waiting Time': mean_ci([np.mean(r['waiting_times_by_eth'][e]) for r in rs if r['waiting_times_by_eth'][e]])[1],
            'Pool Size': mean_ci([r['avg_pool_size_by_eth'][e] for r in rs])[1],
            'L(s) (Left Unmatched)': txt_L,
            '1-F(s)-L(s) (Still in KEP)': still,
            'HLA ClassI': txt_cI,
            'HLA DR': txt_dr,
            'HLA DQ': txt_dq,
        })


    F_tot = [r['F_total'] for r in rs]
    L_tot = [r['L_total'] for r in rs]
    _, txt_F_tot = mean_ci(F_tot); _, txt_L_tot = mean_ci(L_tot)
    arr_tot = np.mean([r['total_arrivals'] for r in rs])
    tx_tot  = np.mean([r['total_transplants'] for r in rs])

    ant_ps=[]; all_ps=[]; epl_ps=[]; cI_ps=[]; dr_ps=[]; dq_ps=[]; wt_ps=[]
    for r in rs:
        ap = [v for e in ACTIVE_ETHCATS for v in r['quality'][('antigen', e)]]
        lp = [v for e in ACTIVE_ETHCATS for v in r['quality'][('allele',  e)]]
        ep = [v for e in ACTIVE_ETHCATS for v in r['quality'][('eplet',   e)]]
        cp = [v for e in ACTIVE_ETHCATS for v in r['quality'][('classI', e)]]
        drp = [v for e in ACTIVE_ETHCATS for v in r['quality'][('DR', e)]]
        dqp = [v for e in ACTIVE_ETHCATS for v in r['quality'][('DQ', e)]]
        wp = [w for e in ACTIVE_ETHCATS for w in r['waiting_times_by_eth'][e]]
        if ap: ant_ps.append(np.mean(ap))
        if lp: all_ps.append(np.mean(lp))
        if ep: epl_ps.append(np.mean(ep))
        if cp: cI_ps.append(np.mean(cp))
        if drp: dr_ps.append(np.mean(drp))
        if dqp: dq_ps.append(np.mean(dqp))
        if wp: wt_ps.append(np.mean(wp))

    _, txt_ant_t = mean_ci(ant_ps); _, txt_all_t = mean_ci(all_ps); _, txt_epl_t = mean_ci(epl_ps)
    _, txt_cI_t = mean_ci(cI_ps); _, txt_dr_t = mean_ci(dr_ps); _, txt_dq_t = mean_ci(dq_ps)
    still_t = round(1 - np.mean(F_tot) - np.mean(L_tot), 3)

    rows.append({
        'Ethnicity': 'Entire Population',
        'Arrivals': round(arr_tot, 2),
        'Transplants': round(tx_tot, 2),
        'F(s) (Matched)': txt_F_tot,
        'HLA(s) Antigen': txt_ant_t,
        'HLA(s) Allele': txt_all_t,
        'HLA(s) Eplets': txt_epl_t,
        'Waiting Time': mean_ci(wt_ps)[1],
        'Pool Size': mean_ci([r['avg_pool_size'] for r in rs])[1],
        'L(s) (Left Unmatched)': txt_L_tot,
        '1-F(s)-L(s) (Still in KEP)': still_t,
        'HLA ClassI': txt_cI_t,
        'HLA DR': txt_dr_t,
        'HLA DQ': txt_dq_t,
    })
    return pd.DataFrame(rows)

tables = {}
for opt_res in RESOLUTIONS:
    print(f"\n========== RESULTS — opt={opt_res} ==========\n")
    tbl = build_results_table(all_results[opt_res])
    tables[opt_res] = tbl
    display(tbl)

out_path = RESULTS_DIR / f'results_databalance_full_10loci_3scenarios{WARMUP_SUFFIX}.xlsx'
with pd.ExcelWriter(out_path) as writer:
    for opt_res, tbl in tables.items():
        tbl.to_excel(writer, sheet_name=f'opt_{opt_res}', index=False)
print(f'\nSaved: {out_path}')


In [ ]:
# RANK TESTS PER ETHNICITY — ALL METRICS, 4 ETHNICITIES (paired across 100 sims)

from scipy.stats import wilcoxon, binomtest

TESTED_ETHCATS = [1, 2, 4, 5]   # data_balance: 4 main ethnicities only

def _mean_or_nan(lst):
    return float(np.mean(lst)) if len(lst) else float('nan')

def _F_eth(r, e):
    return r['F_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')

def _L_eth(r, e):
    return r['L_per_eth'][e] if r['arrivals_by_eth'].get(e, 0) > 0 else float('nan')

METRIC_EXTRACTORS = {
    'F(s) (Matched)':        (_F_eth, lambda r: r['F_total']),
    'L(s) (Left Unmatched)': (_L_eth, lambda r: r['L_total']),
    'HLA(s) Antigen':        (lambda r, e: _mean_or_nan(r['quality'][('antigen', e)]),
                              lambda r:    _mean_or_nan([v for e2 in TESTED_ETHCATS for v in r['quality'][('antigen', e2)]])),
    'HLA(s) Allele':         (lambda r, e: _mean_or_nan(r['quality'][('allele', e)]),
                              lambda r:    _mean_or_nan([v for e2 in TESTED_ETHCATS for v in r['quality'][('allele', e2)]])),
    'HLA(s) Eplets':         (lambda r, e: _mean_or_nan(r['quality'][('eplet', e)]),
                              lambda r:    _mean_or_nan([v for e2 in TESTED_ETHCATS for v in r['quality'][('eplet', e2)]])),
    'Waiting Time':          (lambda r, e: _mean_or_nan(r['waiting_times_by_eth'][e]),
                              lambda r:    _mean_or_nan([w for e2 in TESTED_ETHCATS for w in r['waiting_times_by_eth'][e2]])),
    'HLA ClassI':            (lambda r, e: _mean_or_nan(r['quality'][('classI', e)]),
                              lambda r:    _mean_or_nan([v for e2 in TESTED_ETHCATS for v in r['quality'][('classI', e2)]])),
    'HLA DR':                (lambda r, e: _mean_or_nan(r['quality'][('DR', e)]),
                              lambda r:    _mean_or_nan([v for e2 in TESTED_ETHCATS for v in r['quality'][('DR', e2)]])),
    'HLA DQ':                (lambda r, e: _mean_or_nan(r['quality'][('DQ', e)]),
                              lambda r:    _mean_or_nan([v for e2 in TESTED_ETHCATS for v in r['quality'][('DQ', e2)]])),
}

def _paired_rank_tests(diffs):
   
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try:    _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum())
    n_neg = int((diffs < 0).sum())
    n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {
        'p_wilcoxon':  p_w,
        'p_sign':      p_s,
        'n_pos':       n_pos,
        'n_neg':       n_neg,
        'n_used':      int(len(diffs)),
        'median_diff': float(np.median(diffs)) if len(diffs) else float('nan'),
    }

def rank_test_metric(results_for_scenario, eth_fn, overall_fn, ethcats):
    
    overall_vals = np.array([overall_fn(r) for r in results_for_scenario], dtype=float)
    out = {}
    for e in ethcats:
        eth_vals = np.array([eth_fn(r, e) for r in results_for_scenario], dtype=float)
        mask = ~(np.isnan(eth_vals) | np.isnan(overall_vals))
        diffs = eth_vals[mask] - overall_vals[mask]
        out[e] = _paired_rank_tests(diffs)
    return out

def rank_test_metric_pair(results_A, results_B, eth_fn, ethcats):
    
    n = min(len(results_A), len(results_B))
    out = {}
    for e in ethcats:
        vals_A = np.array([eth_fn(results_A[i], e) for i in range(n)], dtype=float)
        vals_B = np.array([eth_fn(results_B[i], e) for i in range(n)], dtype=float)
        mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
        diffs = vals_A[mask] - vals_B[mask]
        out[e] = _paired_rank_tests(diffs)
    return out


rank_results_all = {}
for opt_res in RESOLUTIONS:
    rank_results_all[opt_res] = {}
    for metric_name, (eth_fn, overall_fn) in METRIC_EXTRACTORS.items():
        rank_results_all[opt_res][metric_name] = rank_test_metric(
            all_results[opt_res], eth_fn, overall_fn, TESTED_ETHCATS)


rank_results_pairwise = {res_A: {} for res_A in RESOLUTIONS}
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B:
            continue
        rank_results_pairwise[res_A][res_B] = {}
        for metric_name, (eth_fn, _) in METRIC_EXTRACTORS.items():
            rank_results_pairwise[res_A][res_B][metric_name] = rank_test_metric_pair(
                all_results[res_A], all_results[res_B], eth_fn, TESTED_ETHCATS)

def _flags(r):
    f = ''
    if not np.isnan(r['p_wilcoxon']) and r['p_wilcoxon'] < 0.05: f += 'W'
    if not np.isnan(r['p_sign'])     and r['p_sign']     < 0.05: f += 'S'
    return f


for opt_res in RESOLUTIONS:
    print(f"\n===== opt={opt_res} — eth-vs-overall flags  [W = Wilcoxon p<.05, S = sign test p<.05] =====")
    header = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
    print(header); print("-" * len(header))
    for m in METRIC_EXTRACTORS:
        row = f"{m:24s}"
        for e in TESTED_ETHCATS:
            row += f"{('[' + _flags(rank_results_all[opt_res][m][e]) + ']'):>14s}"
        print(row)


print("\n\n===== PAIRWISE BETWEEN-RESOLUTION comparisons (paired by sim_id) =====")
print("    [W = Wilcoxon p<.05, S = sign test p<.05]")
seen = set()
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        key = tuple(sorted([res_A, res_B]))
        if key in seen: continue
        seen.add(key)
        a, b = key
        print(f"\n--- {a} vs {b} ---")
        header = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
        print(header); print("-" * len(header))
        for m in METRIC_EXTRACTORS:
            row = f"{m:24s}"
            for e in TESTED_ETHCATS:
                row += f"{('[' + _flags(rank_results_pairwise[a][b][m][e]) + ']'):>14s}"
            print(row)


print(f"\n\n===== F(s) detail (eth-vs-overall: median diff, n used / n+ / n-, p-values) =====")
print(f"{'opt':10s}  {'Ethnicity':14s}  {'diff':>8s}  {'nUsed':>6s}  {'n+':>4s}  {'n-':>4s}  {'p_W':>9s}  {'p_S':>9s}")
for opt_res in RESOLUTIONS:
    for e in TESTED_ETHCATS:
        r = rank_results_all[opt_res]['F(s) (Matched)'][e]
        print(f"{opt_res:10s}  {ETH_LABELS[e]:14s}  {r['median_diff']:>+8.3f}  "
              f"{r['n_used']:>6d}  {r['n_pos']:>4d}  {r['n_neg']:>4d}  {r['p_wilcoxon']:>9.4f}  {r['p_sign']:>9.4f}")
    print()


In [ ]:

from openpyxl import Workbook
from openpyxl.styles import Font

ALPHA = 0.05
METRIC_COLUMNS = list(METRIC_EXTRACTORS.keys())

OTHER_RESS = {opt_res: [r for r in RESOLUTIONS if r != opt_res] for opt_res in RESOLUTIONS}

rank_results_pairwise_overall = {res_A: {} for res_A in RESOLUTIONS}
for res_A in RESOLUTIONS:
    for res_B in RESOLUTIONS:
        if res_A == res_B: continue
        rank_results_pairwise_overall[res_A][res_B] = {}
        for metric_name, (_, overall_fn) in METRIC_EXTRACTORS.items():
            A = all_results[res_A]; B = all_results[res_B]
            n = min(len(A), len(B))
            vals_A = np.array([overall_fn(A[i]) for i in range(n)], dtype=float)
            vals_B = np.array([overall_fn(B[i]) for i in range(n)], dtype=float)
            mask = ~(np.isnan(vals_A) | np.isnan(vals_B))
            rank_results_pairwise_overall[res_A][res_B][metric_name] = _paired_rank_tests(vals_A[mask] - vals_B[mask])

def _pairwise_marks_overall(opt_res, metric):
    others = OTHER_RESS[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        r = rank_results_pairwise_overall[opt_res][other][metric]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks
LABEL_TO_CODE = {ETH_LABELS[e]: e for e in TESTED_ETHCATS}

def _pairwise_marks(opt_res, metric, eth_code):
    if eth_code not in TESTED_ETHCATS: return ''
    others = OTHER_RESS[opt_res]
    marks = ''
    for sym, other in zip(['†', '‡'], others):
        r = rank_results_pairwise[opt_res][other][metric][eth_code]
        if (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA:
            marks += sym
    return marks

wb = Workbook()
wb.remove(wb.active)

for opt_res in RESOLUTIONS:
    ws = wb.create_sheet(f'opt_{opt_res}')
    tbl = tables[opt_res]

    for col_idx, col_name in enumerate(tbl.columns, start=1):
        c = ws.cell(row=1, column=col_idx, value=col_name)
        c.font = Font(bold=True)

    for row_pos, (_, row) in enumerate(tbl.iterrows(), start=2):
        eth_label = row['Ethnicity']
        eth_code = LABEL_TO_CODE.get(eth_label)  

        for col_idx, col_name in enumerate(tbl.columns, start=1):
            val = row[col_name]
            cell = ws.cell(row=row_pos, column=col_idx, value=val)

            if col_name in METRIC_COLUMNS and eth_code in TESTED_ETHCATS:
               
                r = rank_results_all[opt_res][col_name][eth_code]
                bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
                underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
                if bold or underline:
                    cell.font = Font(bold=bold, underline='single' if underline else None)

               
                marks = _pairwise_marks(opt_res, col_name, eth_code)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"'
            if col_name in METRIC_COLUMNS and (eth_code is None):
                marks = _pairwise_marks_overall(opt_res, col_name)
                if marks:
                    if isinstance(val, str):
                        cell.value = val + marks
                    elif isinstance(val, (int, float)) and not (isinstance(val, float) and np.isnan(val)):
                        cell.number_format = f'0.000"{marks}"'


    for col_idx, col_name in enumerate(tbl.columns, start=1):
        col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
        ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)

    foot_row = len(tbl) + 4
    ws.cell(row=foot_row, column=1,
            value='Significance flags — all tested metrics, 4 ethnicities:')
    f_bold = ws.cell(row=foot_row + 1, column=1,
                     value='   bold       = Wilcoxon signed-rank p < 0.05 (eth vs overall, primary)')
    f_und  = ws.cell(row=foot_row + 2, column=1,
                     value='   underlined = sign test p < 0.05 (eth vs overall, robustness)')
    o1, o2 = OTHER_RESS[opt_res]
    ws.cell(row=foot_row + 3, column=1,
            value=f'   †  = Wilcoxon signed-rank p < 0.05 — this resolution ({opt_res}) vs {o1} (paired by sim_id)')
    ws.cell(row=foot_row + 4, column=1,
            value=f'   ‡  = Wilcoxon signed-rank p < 0.05 — this resolution ({opt_res}) vs {o2} (paired by sim_id)')
    ws.cell(row=foot_row + 5, column=1,
            value="   '1-F(s)-L(s)' is derived (1-F-L) and not tested.")
    f_bold.font = Font(bold=True)
    f_und.font  = Font(underline='single')

out_path = RESULTS_DIR / f'results_databalance_full_10loci_3scenarios_significance{WARMUP_SUFFIX}.xlsx'
wb.save(out_path)
print(f'Saved: {out_path}')
print(f'Tested/formatted metric columns: {METRIC_COLUMNS}')
